# Prueba de Amazon Rekognition desde SageMaker

**Objetivo:** verificar que se puede llamar a **Amazon Rekognition** (`DetectLabels`) desde este
notebook de SageMaker y ver qué reconoce en imágenes de terreno marciano (MSL NavCam).

Es un **sondeo inicial sin entrenamiento**: Rekognition genérico devuelve etiquetas amplias
(Nature, Soil, Rock…). Sirve para decidir si conviene un modelo entrenado (Custom Labels) o la
segmentación semántica en SageMaker.

> **Instancia:** este notebook corre bien en `ml.t3.2xlarge` (CPU). Rekognition se ejecuta en el
> servicio de AWS, no en la instancia; aquí solo hacemos llamadas a la API.


## 1. Requisitos

1. **Permisos:** el *rol de ejecución* de este notebook necesita permiso `rekognition:DetectLabels`.
   Si falla con *AccessDenied*, adjunta la política **AmazonRekognitionReadOnlyAccess** al rol
   (IAM → Roles → el rol de SageMaker → Add permissions).
2. **Imágenes:** sube unas imágenes de prueba a la carpeta `rekognition_images/` (junto a este
   notebook). Puedes usar las 8 de `outputs/rekognition_test/images/` del repo.


In [ ]:
import boto3, sagemaker

sess = sagemaker.Session()
region = sess.boto_region_name
try:
    role = sagemaker.get_execution_role()
except Exception:
    role = "(no disponible fuera de SageMaker)"

ident = boto3.client("sts").get_caller_identity()
print("Región         :", region)
print("Cuenta         :", ident["Account"])
print("Identidad      :", ident["Arn"])
print("Rol ejecución  :", role)

## 2. Cargar las imágenes de prueba


In [ ]:
from pathlib import Path

IMAGES_DIR = Path("rekognition_images")   # <-- sube aquí las imágenes de prueba
IMAGES_DIR.mkdir(exist_ok=True)

imgs = sorted(p for p in IMAGES_DIR.iterdir()
              if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
assert imgs, (f"No hay imágenes en {IMAGES_DIR}/. Sube las JPG de "
              "outputs/rekognition_test/images/ del repo y vuelve a ejecutar.")
print(f"{len(imgs)} imágenes:", [p.name for p in imgs])

## 3. Llamar a Rekognition `DetectLabels`


In [ ]:
import pandas as pd

rek = boto3.client("rekognition", region_name=region)
rows = []
for p in imgs:
    try:
        resp = rek.detect_labels(Image={"Bytes": p.read_bytes()},
                                 MaxLabels=20, MinConfidence=40)
    except rek.exceptions.AccessDeniedException:
        raise SystemExit("AccessDenied: el rol de SageMaker no tiene permiso "
                         "rekognition:DetectLabels. Adjunta AmazonRekognitionReadOnlyAccess.")
    labels = resp["Labels"]
    top = ", ".join(f"{l['Name']}({l['Confidence']:.0f}%)" for l in labels[:8])
    n_inst = sum(len(l.get("Instances", [])) for l in labels)
    print(f"\n{p.name}\n  {top}\n  bounding boxes de objetos: {n_inst}")
    for l in labels:
        rows.append({"file": p.name, "label": l["Name"],
                     "confidence": round(l["Confidence"], 1),
                     "instances": len(l.get("Instances", []))})

df = pd.DataFrame(rows)
df.to_csv("rekognition_resultados.csv", index=False)
print("\nGuardado: rekognition_resultados.csv")
df.head(20)

## 4. Visualizar detecciones (bounding boxes) en una imagen


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

p = imgs[0]
im = Image.open(p); W, H = im.size
resp = rek.detect_labels(Image={"Bytes": p.read_bytes()}, MaxLabels=20, MinConfidence=40)

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(im, cmap="gray"); ax.set_title(p.name); ax.axis("off")
n = 0
for l in resp["Labels"]:
    for inst in l.get("Instances", []):
        b = inst["BoundingBox"]; n += 1
        ax.add_patch(patches.Rectangle((b["Left"]*W, b["Top"]*H), b["Width"]*W, b["Height"]*H,
                                       fill=False, edgecolor="red", linewidth=2))
        ax.text(b["Left"]*W, b["Top"]*H - 4, f"{l['Name']} {inst['Confidence']:.0f}%",
                color="red", fontsize=9)
plt.show()
print(f"{n} bounding boxes dibujados. (Rekognition genérico rara vez pone cajas en rocas marcianas.)")

## 5. Interpretación y próximos pasos

- Si Rekognition genérico **no** devuelve etiquetas de roca útiles ni *bounding boxes* de rocas
  individuales (lo más probable), confirma que **no basta el modelo genérico** para este dominio.
- Opciones para darle protagonismo, con evidencia:
  1. **SageMaker Semantic Segmentation** (recomendado): entrenar con las máscaras AI4Mars
     (imagen→máscara) y alimentar las máscaras predichas a nuestro pipeline de cobertura/conteo.
  2. **Rekognition Custom Labels**: entrenar un detector de rocas con *bounding boxes* generados
     desde nuestras máscaras.

El siguiente notebook (`06_...`) montará la opción elegida.
